In [76]:
import os
from pathlib import Path

from dataclasses import dataclass
import polars as pl 
import numpy as np
import pandas as pd

# Deep learning libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from sklearn.preprocessing import StandardScaler

import kaggle_evaluation.default_inference_server

# ============ 1. TensorFlow Configuration ============
def configure_tensorflow():
    """Configure TensorFlow environment safely"""
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
    os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
    
    # Check for GPU availability
    gpus = tf.config.list_physical_devices('GPU')
    if not gpus:
        os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
        print("Using CPU mode")
    else:
        print(f"Detected {len(gpus)} GPU(s)")

configure_tensorflow()

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ============ 2. Paths and Configuration ============
DATA_PATH: Path = Path('/kaggle/input/hull-tactical-market-prediction/')

# ============ 3. Signal Conversion Configuration ============
MIN_SIGNAL: float = 0.0
MAX_SIGNAL: float = 2.0
SIGNAL_MULTIPLIER: float = 400.0

# ============ 4. Model Configuration ============
LOOKBACK_WINDOW: int = 30
BATCH_SIZE: int = 32
EPOCHS: int = 30

# ============ 5. Feature Selection - Based on actual training run ============
# These features were actually used in the successful training run
SELECTED_FEATURES = [
    'M1', 'M10', 'M11', 'M12', 'M13', 
    'E1', 'E10', 'E11', 
    'I1', 'I2', 'I3', 'I4', 
    'P1', 'P10', 'P11', 'P12', 'P13', 
    'V1', 'V10', 'V11', 
    'S1', 'S10', 'S11'
]

print(f"Selected {len(SELECTED_FEATURES)} features")

@dataclass(frozen=True)
class ModelParameters:
    lookback_window: int = LOOKBACK_WINDOW
    batch_size: int = BATCH_SIZE
    epochs: int = EPOCHS
    min_signal: float = MIN_SIGNAL
    max_signal: float = MAX_SIGNAL
    signal_multiplier: float = SIGNAL_MULTIPLIER

params = ModelParameters()

# ============ 6. Data Loading Functions ============
def load_trainset() -> pl.DataFrame:
    """Load and preprocess training dataset"""
    return (
        pl.read_csv(DATA_PATH / "train.csv")
        .rename({'market_forward_excess_returns': 'target'})
        .with_columns(
            pl.exclude('date_id').cast(pl.Float64, strict=False)
        )
    )

def load_testset() -> pl.DataFrame:
    """Load and preprocess test dataset"""
    return (
        pl.read_csv(DATA_PATH / "test.csv")
        .rename({'lagged_forward_returns': 'target'})
        .with_columns(
            pl.exclude('date_id').cast(pl.Float64, strict=False)
        )
    )

def create_feature_dataset(df: pl.DataFrame) -> pl.DataFrame:
    """Create feature dataset with selected features"""
    # Filter to keep only features that actually exist in the dataframe
    vars_to_keep = [col for col in SELECTED_FEATURES if col in df.columns]
    
    # Create basic feature set
    result = df.select(["date_id", "target"] + vars_to_keep)
    
    # Add term spread feature (difference between I2 and I1)
    if all(col in df.columns for col in ["I2", "I1"]):
        result = result.with_columns(
            (pl.col("I2") - pl.col("I1")).alias("term_spread")
        )
    
    # Handle missing values - forward fill then backward fill
    feature_columns = [col for col in result.columns if col not in ['date_id', 'target']]
    for col in feature_columns:
        result = result.with_columns(
            pl.col(col).forward_fill().backward_fill()
        )
    
    # Remove rows that still have null values
    result = result.drop_nulls()
    
    return result

# ============ 7. Simple CNN Model ============
def build_simple_cnn_model(input_shape):
    """Build a simple CNN model for time series prediction"""
    model = models.Sequential([
        layers.Input(shape=input_shape),
        
        # First convolutional block
        layers.Conv1D(filters=32, kernel_size=3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        
        # Second convolutional block
        layers.Conv1D(filters=64, kernel_size=3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling1D(),
        layers.Dropout(0.3),
        
        # Fully connected layers
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.2),
        
        # Output layer
        layers.Dense(1, activation='linear')
    ])
    
    optimizer = keras.optimizers.Adam(learning_rate=0.001)
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
    
    return model

def create_training_callbacks():
    """Create training callbacks"""
    return [
        callbacks.EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True,
            verbose=1
        )
    ]

def convert_ret_to_signal(ret_pred: float, params: ModelParameters) -> float:
    """Convert raw prediction to trading signal"""
    signal = np.clip(
        ret_pred * params.signal_multiplier + 1, 
        params.min_signal, 
        params.max_signal
    )
    return float(signal)

# ============ 8. Sequence Creation ============
def create_sequences_from_dataframe(df: pl.DataFrame, lookback_window: int):
    """Create sequences from DataFrame for time series prediction"""
    # Get feature column names
    feature_cols = [col for col in df.columns if col not in ['date_id', 'target']]
    
    if len(feature_cols) == 0:
        return np.array([]), np.array([]), []
    
    # Convert to numpy arrays
    feature_data = df.select(feature_cols).to_numpy()
    target_data = df.select('target').to_numpy().flatten()
    
    # Create sequences
    X_sequences = []
    y_sequences = []
    
    for i in range(lookback_window, len(feature_data)):
        X_sequences.append(feature_data[i-lookback_window:i])
        y_sequences.append(target_data[i])
    
    if len(X_sequences) == 0:
        return np.array([]), np.array([]), feature_cols
    
    return np.array(X_sequences), np.array(y_sequences), feature_cols

# ============ 9. CNN Predictor Class ============
class SimpleCNNPredictor:
    def __init__(self, lookback_window, feature_cols):
        self.lookback_window = lookback_window
        self.feature_cols = feature_cols
        self.model = None
        self.scaler = StandardScaler()
        self.history_buffer = []
        
    def fit(self, X_train_seq, y_train_seq, validation_split=0.1):
        """Train the CNN model"""
        if len(X_train_seq) == 0:
            print("No training data, skipping training")
            return None
        
        print(f"🔧 Training CNN model, input shape: {X_train_seq.shape}")
        
        # Standardize sequence data
        original_shape = X_train_seq.shape
        X_reshaped = X_train_seq.reshape(-1, X_train_seq.shape[-1])
        X_scaled = self.scaler.fit_transform(X_reshaped)
        X_train_scaled = X_scaled.reshape(original_shape)
        
        # Build model
        input_shape = (self.lookback_window, X_train_scaled.shape[-1])
        self.model = build_simple_cnn_model(input_shape)
        
        # Prepare callbacks
        callbacks_list = create_training_callbacks()
        
        # Split validation set
        split_idx = int(len(X_train_scaled) * (1 - validation_split))
        X_tr = X_train_scaled[:split_idx]
        y_tr = y_train_seq[:split_idx]
        X_val = X_train_scaled[split_idx:]
        y_val = y_train_seq[split_idx:]
        
        print(f"Training set size: {len(X_tr)}, Validation set size: {len(X_val)}")
        
        # Train model
        history = self.model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            batch_size=params.batch_size,
            epochs=params.epochs,
            callbacks=callbacks_list,
            verbose=1
        )
        
        best_val_loss = min(history.history['val_loss'])
        print(f"Training completed, best validation loss: {best_val_loss:.6f}")
        
        return history
    
    def predict_single(self, features: np.ndarray):
        """Make prediction for single data point"""
        if self.model is None:
            print("Model not trained, returning neutral prediction")
            return 0.0
        
        # Add to history buffer
        self.history_buffer.append(features)
        
        # Maintain history buffer length
        if len(self.history_buffer) > self.lookback_window * 2:
            self.history_buffer = self.history_buffer[-self.lookback_window * 2:]
        
        # Prepare sequence
        if len(self.history_buffer) >= self.lookback_window:
            recent_array = np.array(self.history_buffer[-self.lookback_window:])
        else:
            # Pad with zeros if insufficient data
            padding = np.zeros((self.lookback_window - len(self.history_buffer), len(features)))
            recent_array = np.vstack([padding, np.array(self.history_buffer)])
        
        try:
            # Standardize
            recent_reshaped = recent_array.reshape(-1, len(features))
            recent_scaled = self.scaler.transform(recent_reshaped)
            recent_scaled = recent_scaled.reshape(1, self.lookback_window, -1)
            
            # Predict
            y_pred = self.model.predict(recent_scaled, verbose=0)
            return float(y_pred[0, 0])
        except Exception as e:
            print(f"Error during prediction: {e}")
            return 0.0

# ============ 10. Main Prediction Function ============
def predict(test: pl.DataFrame) -> float:
    """Main prediction function - called by Kaggle evaluation API"""
    global predictor, params, FEATURE_COLUMNS
    
    try:
        # Process test data
        test = test.rename({'lagged_forward_returns': 'target'})
        df = create_feature_dataset(test)
        
        # Return neutral signal if data is empty
        if len(df) == 0:
            return 1.0
        
        # Get feature columns
        if not FEATURE_COLUMNS:
            FEATURE_COLUMNS = [col for col in df.columns if col not in ['date_id', 'target']]
        
        if len(FEATURE_COLUMNS) == 0:
            return 1.0
        
        # Extract features
        X_test = df.select(FEATURE_COLUMNS).to_numpy()
        
        if len(X_test) == 0:
            return 1.0
        
        # Use predictor for prediction
        if predictor is None:
            return 1.0
        
        # Make prediction
        raw_pred = predictor.predict_single(X_test[0])
        
        # Convert to trading signal
        signal = convert_ret_to_signal(raw_pred, params)
        
        return signal
        
    except Exception as e:
        # Return neutral signal on any error
        print(f"Prediction error: {str(e)[:100]}")
        return 1.0

# ============ 11. Main Program ============
def main():
    """Main function"""
    global predictor, FEATURE_COLUMNS
    
    print("=" * 60)
    print("Hull Tactical Market Prediction - Simple CNN Model")
    print("=" * 60)
    
    # 1. Load data
    print("\n1. Loading data...")
    train_df = load_trainset()
    test_df = load_testset()
    
    print(f"  Training data shape: {train_df.shape}")
    print(f"  Test data shape: {test_df.shape}")
    
    # 2. Create feature dataset
    print("\n2. Creating feature dataset...")
    train_features = create_feature_dataset(train_df)
    test_features = create_feature_dataset(test_df)
    
    print(f"  Training features shape: {train_features.shape}")
    print(f"  Test features shape: {test_features.shape}")
    
    # 3. Get feature columns
    FEATURE_COLUMNS = [col for col in train_features.columns if col not in ['date_id', 'target']]
    print(f"\n3. Actual features used: {len(FEATURE_COLUMNS)}")
    
    # 4. Prepare training sequences
    print(f"\n4. Preparing training sequences (lookback: {params.lookback_window})...")
    X_train_seq, y_train_seq, feature_cols = create_sequences_from_dataframe(
        train_features, params.lookback_window
    )
    
    if len(X_train_seq) == 0:
        print("Cannot create training sequences, insufficient data")
        return
    
    print(f"  Training sequences shape: {X_train_seq.shape}")
    print(f"  Target sequences shape: {y_train_seq.shape}")
    
    # 5. Initialize and train CNN model
    print("\n5. Initializing and training CNN model...")
    predictor = SimpleCNNPredictor(params.lookback_window, feature_cols)
    
    print("Starting CNN model training...")
    history = predictor.fit(X_train_seq, y_train_seq, validation_split=0.1)
    print("CNN model training completed")
    
    # 6. Initialize historical data
    print("\n6. Initializing historical data...")
    if len(train_features) >= params.lookback_window:
        last_train_features = train_features.tail(params.lookback_window).select(FEATURE_COLUMNS).to_numpy()
        
        # Populate history buffer
        for i in range(len(last_train_features)):
            predictor.history_buffer.append(last_train_features[i])
        
        print(f"  History buffer initialized, size: {len(predictor.history_buffer)}")
    else:
        print(f"  Insufficient training data for lookback window")
    
    # 7. Start evaluation server
    print("\n7. Starting evaluation server...")
    inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)
    
    if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
        print("Entering competition rerun mode, starting server...")
        inference_server.serve()
    else:
        print("Local test mode, starting local gateway...")
        inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))

# ============ 12. Program Entry Point ============
if __name__ == "__main__":
    # Define global variables
    predictor = None
    FEATURE_COLUMNS = []
    
    try:
        main()
    except Exception as e:
        print(f"\nProgram execution error: {str(e)}")

Using CPU mode
Selected 23 features
Hull Tactical Market Prediction - Simple CNN Model

1. Loading data...
  Training data shape: (9021, 98)
  Test data shape: (10, 99)

2. Creating feature dataset...
  Training features shape: (9021, 26)
  Test features shape: (10, 26)

3. Actual features used: 24

4. Preparing training sequences (lookback: 30)...
  Training sequences shape: (8991, 30, 24)
  Target sequences shape: (8991,)

5. Initializing and training CNN model...
Starting CNN model training...
🔧 Training CNN model, input shape: (8991, 30, 24)
Training set size: 8091, Validation set size: 900
Epoch 1/30
253/253 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - loss: 0.2108 - mae: 0.3347 - val_loss: 0.0076 - val_mae: 0.0709
Epoch 2/30
253/253 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.0232 - mae: 0.1167 - val_loss: 0.0051 - val_mae: 0.0616
Epoch 3/30
253/253 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.0074 - mae: 0.0641 - val_loss: 0.0029 - val_mae: 0.0437
Epoch 4/30
253/253 ━━━━━━━━━━━━━━━━━━━━ 2s 